**Sample ID**: 27_base_US_ToolShift

**Query**:

Create a shared folder.

**DB Type**: Base Case

**Case Description**:

The Google Chat space, named "pod leads" exists. No message has been sent in the "pod leads" Google chat space. There is no existing folder named "Colab files for AI agents" in the user's Google Drive.

```
<multiturn info>
[turn 1]: Folder Name: Colab files for AI agents (Information Gathering)
[turn 2]: Notification Request: Announce the folder creation in a Google chat space. (Follow Up Request)
[turn 3]: Chat Space Name: pod leads (Information Gathering)
</multiturn info>
```

```
<tools>
[turn 0]: gdrive
[turn 2]: google_chat
</tools>
```

**Global/Context Variables:**


**APIs:**

- google_chat
- gdrive


# Set Up

## Download relevant files

In [1]:
import io
import os
import sys
import zipfile
import shutil
import re
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Version to download
VERSION = "0.1.4"  # This will be replaced dynamically

# Define paths
CONTENT_DIR = '/content'
APIS_DIR = os.path.join(CONTENT_DIR, 'APIs')
DBS_DIR = os.path.join(CONTENT_DIR, 'DBs')
SCRIPTS_DIR = os.path.join(CONTENT_DIR, 'Scripts')
FC_DIR = os.path.join(CONTENT_DIR, 'Schemas')
ZIP_PATH = os.path.join(CONTENT_DIR, f'APIs_V{VERSION}.zip')

# Google Drive Folder ID where versioned APIs zip files are stored
APIS_FOLDER_ID = '1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4'

# List of items to extract from the zip file
ITEMS_TO_EXTRACT = ['APIs/', 'DBs/', 'Scripts/', 'Schemas/']

# Clean up existing directories and files
for path in [APIS_DIR, DBS_DIR, SCRIPTS_DIR, FC_DIR, ZIP_PATH]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

# Authenticate and create the drive service
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Helper function to download a file from Google Drive
def download_drive_file(service, file_id, output_path, file_name=None, show_progress=True):
    """Downloads a file from Google Drive"""
    destination = output_path
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(destination, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if show_progress:
                print(f"Download progress: {int(status.progress() * 100)}%")

# 1. List files in the specified APIs folder
print(f"Searching for APIs zip file with version {VERSION} in folder: {APIS_FOLDER_ID}...")
apis_file_id = None

try:
    query = f"'{APIS_FOLDER_ID}' in parents and trashed=false"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get('files', [])
    for file in files:
        file_name = file.get('name', '')
        if file_name.lower() == f'apis_v{VERSION.lower()}.zip':
            apis_file_id = file.get('id')
            print(f"Found matching file: {file_name} (ID: {apis_file_id})")
            break

except Exception as e:
    print(f"An error occurred while listing files in Google Drive: {e}")

if not apis_file_id:
    print(f"Error: Could not find APIs zip file with version {VERSION} in the specified folder.")
    sys.exit("Required APIs zip file not found.")

# 2. Download the found APIs zip file
print(f"Downloading APIs zip file with ID: {apis_file_id}...")
download_drive_file(drive_service, apis_file_id, ZIP_PATH, file_name=f'APIs_V{VERSION}.zip')

# 3. Extract specific items from the zip file to /content
print(f"Extracting specific items from {ZIP_PATH} to {CONTENT_DIR}...")
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()

        for member in zip_contents:
            extracted = False
            for item_prefix in ITEMS_TO_EXTRACT:
                if member == item_prefix or member.startswith(item_prefix):
                    zip_ref.extract(member, CONTENT_DIR)
                    extracted = True
                    break

except zipfile.BadZipFile:
    print(f"Error: The downloaded file at {ZIP_PATH} is not a valid zip file.")
    sys.exit("Invalid zip file downloaded.")
except Exception as e:
    print(f"An error occurred during extraction: {e}")
    sys.exit("Extraction failed.")

# 4. Clean up
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# 5. Add APIs to path
if os.path.exists(APIS_DIR):
    sys.path.append(APIS_DIR)
else:
    print(f"Error: APIS directory not found at {APIS_DIR} after extraction. Cannot add to path.")

# 6. Quick verification
# Check for the presence of the extracted items
verification_paths = [APIS_DIR, DBS_DIR, SCRIPTS_DIR]
all_present = True
print("\nVerifying extracted items:")
for path in verification_paths:
    if os.path.exists(path):
        print(f"✅ {path} is present.")
    else:
        print(f"❌ {path} is MISSING!")
        all_present = False

if all_present:
    print(f"\n✅ Setup complete! Required items extracted to {CONTENT_DIR}.")
else:
    print("\n❌ Setup failed! Not all required items were extracted.")
os.chdir(CONTENT_DIR)

Searching for APIs zip file with version 0.1.4 in folder: 1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4...
Found matching file: APIs_V0.1.4.zip (ID: 1TnAaWGfVrMxWTilyhy46-Aue_bh0XkNk)
Download progress: 100%
Extracting specific items from /content/APIs_V0.1.4.zip to /content...

Verifying extracted items:
✅ /content/APIs is present.
✅ /content/DBs is present.
✅ /content/Scripts is present.

✅ Setup complete! Required items extracted to /content.


## Install Dependencies and Clone Repositories

In [2]:
!pip install -r /content/APIs/requirements.txt

## Import APIs and initiate DBs

In [3]:
# proto_ignore
import random
import sys
import uuid
import secrets

# Import libraries to ensure all initializations by the python libraries are complete


def patch_randomness(seed=42):
    rng = random.Random(seed)
    random.seed(seed)

    # Patch uuid.uuid4
    def deterministic_uuid4():
        return uuid.UUID(int=rng.getrandbits(128))
    sys.modules['uuid'].uuid4 = deterministic_uuid4

    # Patch secrets to use the same deterministic random generator
    class DeterministicRandom:
        def randbelow(self, n):
            return rng.randrange(n)
        def choice(self, seq):
            return rng.choice(seq)
        def randbits(self, k):
            return rng.getrandbits(k)
        def randint(self, a, b):
            return rng.randint(a, b)
    sys.modules['secrets'] = DeterministicRandom()

patch_randomness()

In [4]:
import google_chat
import gdrive

# --- Load simulation databases ---
google_chat.SimulationEngine.db.load_state("/content/DBs/GoogleChatDefaultDB.json")
gdrive.SimulationEngine.db.load_state("/content/DBs/GDriveDefaultDB.json")


# --- Google Chat Setup ---
print("\nSetting up Google Chat...")

# Create a space
print("Creating Google Chat space 'pod leads'...")
chat_space_data = {
    "space": {
        "displayName": "pod leads",
        "spaceType": "SPACE"
    }
}
pod_leads_chat_space = google_chat.create_space(space=chat_space_data['space'])
SPACE_RESOURCE_NAME = pod_leads_chat_space.get('name')
print(f"Google Chat space 'pod leads' created with name: {SPACE_RESOURCE_NAME}")

print("Google Chat setup complete.")

# Ensure SPACE_DISPLAY_NAME is also set for consistency if used elsewhere
SPACE_DISPLAY_NAME = "pod leads"
if not SPACE_RESOURCE_NAME:
    print(f"Warning: Could not resolve resource name for '{SPACE_DISPLAY_NAME}'. "
          f"Subsequent chat actions should verify the parent explicitly.")
else:
    print(f"Using Chat space: {SPACE_DISPLAY_NAME} -> {SPACE_RESOURCE_NAME}")


Setting up Google Chat...
Creating Google Chat space 'pod leads'...
Google Chat space 'pod leads' created with name: spaces/SPACE_2
Google Chat setup complete.
Using Chat space: pod leads -> spaces/SPACE_2


# Initial Assertion

1. Assert that the folder "Colab files for AI agents"  does not exist in Google Drive.
2. Assert that the space "pod leads" exists.
3. Assert that no message has been sent to "pod leads" Google Chat space.

In [5]:
from Scripts.assertions_utils import *
import google_chat
import gdrive

# Constants
folder_name = "Colab files for AI agents"
space_display_name = "pod leads"

# --- Assertion 1: Assert that the folder 'Colab files for AI agents' does not exist in Google Drive. ---
folder_list_response = {}
gdrive_error = None
found_folders = []

try:
    list_files_params = {
        'q': f"name = '{folder_name}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false",
        'spaces': 'drive',
    }
    folder_list_response = gdrive.list_user_files(**list_files_params)
    found_folders = folder_list_response.get('files', [])
except Exception as e:
    gdrive_error = str(e)

assertion_condition_1 = len(found_folders) == 0

assertion_message_1 = "Assertion 1 Failed: "
if gdrive_error:
    assertion_message_1 += f"Google Drive API call failed: {gdrive_error}."
else:
    folder_names = [f.get("name", "Unnamed") for f in found_folders[:3]]
    assertion_message_1 += (
        f"Expected folder '{folder_name}' not to exist, but found {len(found_folders)} instance(s). "
        f"Examples: {folder_names}"
    )

assert assertion_condition_1, assertion_message_1

# --- Assertion 2: Assert that the Google Chat space 'pod leads' exists. ---
pod_leads_space_found = False
chat_spaces_api_error = None
page_token = None
all_spaces_checked = []
pod_leads_space_resource_name = ""

try:
    while True:
        list_spaces_params = {
            'pageSize': 200,
            'pageToken': page_token
        }
        space_list_response = google_chat.list_spaces(**list_spaces_params)
        spaces = space_list_response.get('spaces', [])
        all_spaces_checked.extend(spaces)

        for space in spaces:
            if isinstance(space, dict) and compare_strings(space.get('displayName'), space_display_name):
                pod_leads_space_found = True
                pod_leads_space_resource_name = space.get("name")
                break

        if pod_leads_space_found:
            break

        page_token = space_list_response.get('nextPageToken')
        if not page_token:
            break

except Exception as e:
    chat_spaces_api_error = str(e)

assertion_condition_2 = pod_leads_space_found

assertion_message_2 = "Assertion 2 Failed: "
if chat_spaces_api_error:
    assertion_message_2 += f"Google Chat API call failed: {chat_spaces_api_error}."
elif not pod_leads_space_found:
    checked_names = [s.get("displayName", "Unnamed") for s in all_spaces_checked[:3]]
    assertion_message_2 += (
        f"Expected space '{space_display_name}' to exist but it was not found. "
        f"Checked spaces: {checked_names}"
    )

assert assertion_condition_2, assertion_message_2

# --- Assertion 3: Assert that no message has been sent to "pod leads" Google Chat space. ---
messages_exist = False
chat_messages_api_error = None
messages = []

if pod_leads_space_resource_name:
    try:
        messages_response = google_chat.list_messages(parent=pod_leads_space_resource_name)
        # Check if 'messages' key exists and if its list is not empty
        messages = messages_response.get('messages', [])
        messages_exist = bool(messages)
    except Exception as e:
        chat_messages_api_error = str(e) # Capture API error

assertion_condition_3 = not messages_exist

assertion_message_3 = "Assertion 3 Failed: "
if chat_messages_api_error:
    assertion_message_3 += f"Google Chat API call failed while listing messages: {chat_messages_api_error}."
elif messages_exist:
    assertion_message_3 += f"Messages were found in the Google Chat space '{space_display_name}' ({pod_leads_space_resource_name}), but none were expected."
else:
     # This case should not be reached if assertion_condition_3 is False, but included for completeness
     assertion_message_3 += f"Unexpected state: messages_exist is False but assertion_condition_3 is False."


assert assertion_condition_3, assertion_message_3

# Action

**Simulated User**: Create a shared folder.

In [6]:
# proto_ignore
import gdrive

**Action Agent**: I can help with that. What would you like to name the folder? I will make it viewable by anyone with the link.

**Simulated User**: The folder name is "Colab files for AI agents".

In [7]:
gdrive.create_file_or_folder(body={'mimeType': 'application/vnd.google-apps.folder', 'name': 'Colab files for AI agents'})

{'kind': 'drive#file',
 'id': 'file_3',
 'driveId': '',
 'name': 'Colab files for AI agents',
 'mimeType': 'application/vnd.google-apps.folder',
 'parents': [],
 'createdTime': '2025-03-14T00:00:00Z',
 'modifiedTime': '2025-03-14T00:00:00Z',
 'trashed': False,
 'starred': False,
 'owners': ['john.doe@gmail.com'],
 'size': '0',
 'md5Checksum': '',
 'sha1Checksum': '',
 'sha256Checksum': '',
 'imageMediaMetadata': {},
 'videoMediaMetadata': {},
 'permissions': [{'id': 'permission_file_3',
   'role': 'owner',
   'type': 'user',
   'emailAddress': 'john.doe@gmail.com'}],
 'enforceSingleParent': False,
 'ignoreDefaultVisibility': False,
 'keepRevisionForever': False,
 'ocrLanguage': '',
 'supportsAllDrives': False,
 'supportsTeamDrives': False,
 'useContentAsIndexableText': False,
 'includePermissionsForView': '',
 'includeLabels': '',
 'revisionSettings': {'keepForever': False},
 'ocrMetadata': {},
 'indexableText': '',
 'additionalPermissions': [],
 'labels': []}

In [8]:
gdrive.create_permission(body={'role': 'viewer', 'type': 'anyone'}, fileId='file_3')

{'kind': 'drive#permission',
 'id': 'permission_1',
 'role': 'viewer',
 'type': 'anyone',
 'emailAddress': '',
 'domain': '',
 'allowFileDiscovery': False,
 'expirationTime': ''}

**Action Agent**: I have created the folder named 'Colab files for AI agents' and made it accessible to anyone with the link.

Is there anything else I can help you with?

**Simulated User**: Yes, now announce the folder creation in a Google chat space.

In [9]:
# proto_ignore
import google_chat

**Action Agent**: I can do that. What is the name of the space you would like to announce it in?

**Simulated User**: The chat space is "pod leads".

In [10]:
google_chat.list_spaces()

{'spaces': [{'name': 'spaces/SPACE_2',
   'spaceType': 'SPACE',
   'displayName': 'pod leads',
   'externalUserAllowed': False,
   'createTime': '2025-11-08T15:07:19.995353Z',
   'importMode': False,
   'singleUserBotDm': False}],
 'nextPageToken': ''}

In [11]:
google_chat.create_message(parent="spaces/SPACE_2",message_body={'text': "A common driver folder 'Colab files for AI agents' has been created. "})

{'name': 'spaces/SPACE_2/messages/2',
 'text': "A common driver folder 'Colab files for AI agents' has been created. ",
 'attachment': [],
 'createTime': '2025-11-08T15:07:20.072711Z',
 'thread': {},
 'requestId': None,
 'sender': {'name': 'users/USER123', 'type': 'HUMAN'}}

**Action Agent**: I have posted the message to the 'pod leads' space about folder creation.

# Final Assertion

1. Assert that the folder "Colab files for AI agents" exist in Google Drive.
2. Assert that a message has been sent to "pod leads" Google Chat space.

In [14]:
from Scripts.assertions_utils import *
import google_chat
import gdrive

# --- Constants ---
folder_name = "Colab files for AI agents"
space_display_name = "pod leads"

# --- Assertion 1: Assert that the folder "Colab files for AI agents" exist in Google Drive. ---
api_error_1 = None
found_folders = []

try:
    list_files_params = {
        'q': f"name = '{folder_name}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false",
        'spaces': 'drive',
    }
    folder_list_response = gdrive.list_user_files(**list_files_params)
    found_folders = folder_list_response.get('files', [])
except Exception as e:
    api_error_1 = str(e)

assertion_condition_1 = len(found_folders) == 1
assertion_message_1 = (
    f"Assertion 1 Failed: Expected exactly one folder named '{folder_name}', but found {len(found_folders)}."
)
if api_error_1:
    assertion_message_1 += f" Google Drive API call failed: {api_error_1}"
assert assertion_condition_1, assertion_message_1

# --- Assertion 1b: Check if the folder is shared properly (anyone viewer/writer). ---
perm_api_error = None
permissions = []
shared_ok = False

try:
    folder_id = found_folders[0].get('id')
    perm_response = gdrive.list_permissions(fileId=folder_id)
    permissions = perm_response.get('permissions', [])
    shared_ok = any(
        p.get('type') == 'anyone' and p.get('role') in ['viewer', 'writer']
        for p in permissions
    )
except Exception as e:
    perm_api_error = str(e)

assertion_condition_1b = shared_ok
assertion_message_1b = (
    f"Assertion 1b Failed: Folder '{folder_name}' exists but is not shared properly "
    f"(expected permission type 'anyone' and role 'viewer' or 'writer')."
)
if perm_api_error:
    assertion_message_1b += f" Google Drive API call failed: {perm_api_error}"
assert assertion_condition_1b, assertion_message_1b

# --- Assertion 2: Assert that a message has been sent on "pod leads" space. ---
message_sent = False
pod_leads_space_resource_name = ""
page_token = None
api_error_2 = None

try:
    # Find the space resource name
    while True:
        list_spaces_params = {
            'pageSize': 200,
            'pageToken': page_token
        }
        space_list_response = google_chat.list_spaces(**list_spaces_params)
        spaces = space_list_response.get('spaces', [])

        for space in spaces:
            if isinstance(space, dict) and compare_strings(space.get('displayName'), space_display_name):
                pod_leads_space_resource_name = space.get("name")
                break

        if pod_leads_space_resource_name:
            break

        page_token = space_list_response.get('nextPageToken')
        if not page_token:
            break

    if pod_leads_space_resource_name:
        # List messages in the space
        message_list_response = google_chat.list_messages(parent=pod_leads_space_resource_name)
        messages = message_list_response.get('messages', [])
        # Minor fix: check message text content for folder creation or sharing
        if messages:
            lname = folder_name.lower()
            message_sent = any(
                (lname in msg.get('text', '').lower()) and
                (('created' in msg.get('text', '').lower()) or ('share' in msg.get('text', '').lower()))
                for msg in messages
            )
    else:
        # If space not found, rely on message_sent being False
        pass

except Exception as e:
    # If any API call fails, message_sent should be False
    message_sent = False
    api_error_2 = str(e)

assertion_condition_2 = message_sent
assertion_message_2 = (
    f"Assertion 2 Failed: No appropriate message was found in the Google Chat space '{space_display_name}' "
    f"({pod_leads_space_resource_name}). Expected a message mentioning '{folder_name}' and indicating creation/sharing."
)
if api_error_2:
     assertion_message_2 += f" Google Chat API call failed: {api_error_2}"

assert assertion_condition_2, assertion_message_2
